In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from multiprocessing import Pool, current_process
import time

In [2]:
import pickle

with open("Corr_len_cytnx.pkl", "rb") as file:
    loaded_data = pickle.load(file)
    
Corr_len = loaded_data['Corr_len']

In [3]:
def get_W(beta, J=1, h=0):
    """
    Calculate the matrix W based on beta, J, and h.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength (default is 1).
        h (float): External field strength (default is 0).

    Returns:
        torch.Tensor: Matrix W.
    """
    sq_cosh = np.sqrt(np.cosh(beta * J))
    sq_sinh = np.sqrt(np.sinh(beta * J))
    W = torch.tensor([
        [sq_cosh, sq_sinh],
        [sq_cosh, -sq_sinh]
    ])
    return W


#可能Wh是正確的包含h的作法
def get_Wh(beta, J, h):
    """
    Calculate the matrix Wh incorporating the external field h.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength.
        h (float): External field strength.

    Returns:
        torch.Tensor: Matrix Wh.
    """
    cosh = np.cosh(beta * J)
    sinh = np.sinh(beta * J)

    Wh = torch.tensor([
        [np.sqrt(cosh) * np.exp(beta * h / 2), np.sqrt(sinh) * np.exp(beta * h / 2)],
        [np.sqrt(cosh) * np.exp(-beta * h / 2), -np.sqrt(sinh) * np.exp(-beta * h / 2)]
    ])
    return Wh


# 上左右下
def get_T_bare(beta, J=1, h=0, get_W=get_W):
    """
    Calculate the bare transfer matrix T_bare.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength (default is 1).
        h (float): External field strength (default is 0).
        get_W (callable): Function to calculate W or Wh (default is get_W).

    Returns:
        torch.Tensor: Bare transfer matrix T_bare.
    """
    W = get_W(beta, J=J, h=h)
    T_bare = torch.einsum('ai,aj,ak,al->ijkl', W, W, W, W).to(torch.float32)
    return T_bare

def merge_y(Tup, Tdn, combine=False, trace=False):
    if trace:
        Tmerge = torch.einsum('abcd,defa->becf', Tup, Tdn)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.contiguous().view(shape[0] * shape[1], shape[2] * shape[3])
    else:
        Tmerge = torch.einsum('abcd,defg->abecfg', Tup, Tdn)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.contiguous().view(shape[0], shape[1] * shape[2], shape[3] * shape[4], shape[5])
            
    return Tmerge


def merge_y_truncate(Tup, Tdn, dcut):
    if ((Tup.shape[1] * Tdn.shape[1]) < dcut):
        return merge_y(Tup, Tdn, True)
    Tmerge_pure = torch.einsum('aAbc,cBef,aCbd,dDef->ABCD', Tup, Tdn, Tup, Tdn)
    Tmerge_pure_shape = Tmerge_pure.shape
    Tmerge_pure = Tmerge_pure.contiguous().view(
        Tmerge_pure_shape[0] * Tmerge_pure_shape[1],Tmerge_pure_shape[2] * Tmerge_pure_shape[3]
    )
    U, S, V = torch.svd(Tmerge_pure)
    U = U[:,:dcut]
    U = U.view(Tmerge_pure_shape[0], Tmerge_pure_shape[1], dcut)
    Tmerge = torch.einsum('Aace,ebdD,cdC,abB->ABCD', Tup, Tdn, U, U)
    return Tmerge

def merge_x(TL, TR, combine=False, trace=False):
    if trace:
        Tmerge = torch.einsum('aAca,ecBe->AB', TL, TR)
    else:
        Tmerge = torch.einsum('BCaF,AaDE->ABCDEF', TL, TR)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.contiguous().view(shape[0] * shape[1], shape[2], shape[3], shape[4] * shape[5])
            
    return Tmerge

def merge_x_truncate(TL, TR, dcut):
    if ((TL.shape[0] * TR.shape[0]) < dcut):
        return merge_x(TL, TR, True)
    Tmerge_pure = torch.einsum('abcd,ecfg,hbid,jifg->aehj', TL, TR, TL, TR)
    Tmerge_pure_shape = Tmerge_pure.shape
    Tmerge_pure = Tmerge_pure.contiguous().view(
        Tmerge_pure_shape[0] * Tmerge_pure_shape[1],Tmerge_pure_shape[2] * Tmerge_pure_shape[3]
    )
    U, S, V = torch.svd(Tmerge_pure)
    U = U[:,:dcut]
    U = U.view(Tmerge_pure_shape[0], Tmerge_pure_shape[1], dcut)
    Tmerge = torch.einsum('abcd,ecfg,aeh,dgi->hbfi', TL, TR, U, U)
    return Tmerge

In [4]:
MaxL = 11
Temp = np.linspace(2.2,2.35,100)
Tc = 2/np.log(1+np.sqrt(2))
dbeta = 1e-2
dcut = 30
num_processes = 2

In [5]:
%%time

def init_worker(process_id):
    """初始化每個進程，設置唯一名稱"""
    current_process().name = f"{process_id}"

def compute_Cv_for_temp_cuda(i, temp):
    process_name = int(current_process().name[-1])
    print(f'Start {i}: {temp:.3f} {process_name} {current_process().name}\n')
    st = time.time()
    beta = 1 / temp
    Cv_i = np.zeros(MaxL + 1)
    W = np.zeros((MaxL + 1, 2))
    
    T_bare = get_T_bare(beta).cuda(process_name)
    TL = T_bare
    for j in range(2, MaxL + 1):
        TLx = merge_x_truncate(TL, TL, dcut)
        TL_Trace = merge_y(TLx, TLx, True, True)
        eigvals = torch.linalg.eigvalsh(TL_Trace).cpu()
        Cv_i[j] -= 2 * torch.log(eigvals[-1]).item()
        W[j, :] = eigvals[-2:]
        TL = merge_y_truncate(TLx, TLx, dcut)
        TL = TL / torch.mean(torch.abs(TL))
    
    E = -np.log(W)
    Corr_len = 1 / (E[:, -2] - E[:, -1])
    
    print(f'End {i}: {temp:.3f} Use time: {(time.time() - st) :.2f}')
    return i, Cv_i, Corr_len

Cv = np.zeros((MaxL + 1, len(Temp)))
Corr_len_matrix = np.zeros((MaxL + 1, len(Temp)))

with Pool(processes=num_processes, initializer=init_worker, initargs=(0,)) as pool:
    init_args = [(i,) for i in range(num_processes)]
    for i, args in enumerate(init_args):
        pool.apply_async(init_worker, args=(args[0],))
    results = pool.starmap(compute_Cv_for_temp_cuda, [(i, temp) for i, temp in enumerate(Temp)])

# 將結果合併回主矩陣
for i, Cv_i, Corr_len in results:
    Cv[:, i] = Cv_i
    Corr_len_matrix[:, i] = Corr_len

# Save results and parameters using pickle
output_data = {
    "MaxL": MaxL,
    "Temp": Temp.tolist(),
    "Tc": Tc,
    "dbeta": dbeta,
    "Cv": Cv.tolist(),
    "Corr_len": Corr_len_matrix.tolist(),
    "dcut": dcut
}

with open(f"Cv_results_{dcut}_{dbeta}.pkl", "wb") as f:
    pickle.dump(output_data, f)

print("Results saved to Cv_results.pkl")

Start 0: 2.200 1 1

Start 13: 2.220 0 0



<timed exec>:20: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
<timed exec>:20: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
<timed exec>:24: RuntimeWarning: divide by zero encountered in log
<timed exec>:25: RuntimeWarning: invalid value encountered in subtract
<timed exec>:25: RuntimeWarning: divide by zero encountered in divide


End 0: 2.200 Use time: 3.17
Start 1: 2.202 1 1



<timed exec>:24: RuntimeWarning: divide by zero encountered in log
<timed exec>:25: RuntimeWarning: invalid value encountered in subtract


End 13: 2.220 Use time: 3.31
Start 14: 2.221 0 0

End 1: 2.202 Use time: 2.06
Start 2: 2.203 1 1

End 14: 2.221 Use time: 2.19
Start 15: 2.223 0 0

End 2: 2.203 Use time: 2.07
Start 3: 2.205 1 1

End 15: 2.223 Use time: 2.18
Start 16: 2.224 0 0

End 3: 2.205 Use time: 2.07
Start 4: 2.206 1 1



<timed exec>:25: RuntimeWarning: divide by zero encountered in divide


End 16: 2.224 Use time: 2.19
Start 17: 2.226 0 0

End 4: 2.206 Use time: 2.07
Start 5: 2.208 1 1

End 17: 2.226 Use time: 2.20
Start 18: 2.227 0 0

End 5: 2.208 Use time: 2.09
Start 6: 2.209 1 1

End 18: 2.227 Use time: 2.21
Start 19: 2.229 0 0

End 6: 2.209 Use time: 2.10
Start 7: 2.211 1 1

End 19: 2.229 Use time: 2.22
Start 20: 2.230 0 0

End 7: 2.211 Use time: 2.13
Start 8: 2.212 1 1

End 20: 2.230 Use time: 2.22
Start 21: 2.232 0 0

End 8: 2.212 Use time: 2.12
Start 9: 2.214 1 1

End 21: 2.232 Use time: 2.20
Start 22: 2.233 0 0

End 9: 2.214 Use time: 2.16
Start 10: 2.215 1 1

End 22: 2.233 Use time: 2.25
Start 23: 2.235 0 0

End 10: 2.215 Use time: 2.15
Start 11: 2.217 1 1

End 23: 2.235 Use time: 2.25
Start 24: 2.236 0 0

End 11: 2.217 Use time: 2.15
Start 12: 2.218 1 1

End 24: 2.236 Use time: 2.28
Start 25: 2.238 0 0

End 12: 2.218 Use time: 2.18
Start 26: 2.239 1 1

End 25: 2.238 Use time: 2.30
Start 39: 2.259 0 0

End 26: 2.239 Use time: 2.35
Start 27: 2.241 1 1

End 39: 2.2

In [ ]:
%%time

def compute_Cv_for_temp(i, temp):
    print(f'Start {i}: {temp:.3f}')
    st = time.time()
    beta = 1 / temp
    Cv_i = np.zeros(MaxL + 1)
    W = np.zeros((MaxL + 1, 2))
    
    T_bare = get_T_bare(beta)
    TL = T_bare
    for j in range(2, MaxL + 1):
        TLx = merge_x_truncate(TL, TL, dcut)
        TL_Trace = merge_y(TLx, TLx, True, True)
        eigvals = torch.linalg.eigvalsh(TL_Trace).cpu()
        Cv_i[j] -= 2 * torch.log(eigvals[-1]).item()
        W[j, :] = eigvals[-2:]
        TL = merge_y_truncate(TLx, TLx, dcut)
        TL = TL / torch.mean(torch.abs(TL))
    
    E = -np.log(W)
    Corr_len = 1 / (E[:, -2] - E[:, -1])
    
    print(f'End {i}: {temp:.3f} Use time: {(time.time() - st) :.2f}')
    return i, Cv_i, Corr_len

Cv = np.zeros((MaxL + 1, len(Temp)))
Corr_len_matrix = np.zeros((MaxL + 1, len(Temp)))

with Pool(processes=3) as pool:  # 4 個進程，根據 CPU 核心數調整
    results = pool.starmap(compute_Cv_for_temp, [(i, temp) for i, temp in enumerate(Temp)])

# 將結果合併回主矩陣
for i, Cv_i, Corr_len in results:
    Cv[:, i] = Cv_i
    Corr_len_matrix[:, i] = Corr_len

# Save results and parameters using pickle
output_data = {
    "MaxL": MaxL,
    "Temp": Temp.tolist(),
    "Tc": Tc,
    "dbeta": dbeta,
    "Cv": Cv.tolist(),
    "Corr_len": Corr_len_matrix.tolist(),
    "dcut": dcut
}

with open(f"Cv_results_{dcut}_{dbeta}.pkl", "wb") as f:
    pickle.dump(output_data, f)

print("Results saved to Cv_results.pkl")

Start 0: 2.200
Start 9: 2.214
Start 18: 2.227


<timed exec>:15: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
<timed exec>:15: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
<timed exec>:15: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
